# Modelado: Regresión Logística (Corregido - Sin Data Leakage)

Este notebook se enfoca en el modelado del dataset Bank Marketing usando **Regresión Logística**. 

## 1. Introducción al Modelo

**¿Qué es la Regresión Logística?**
Es un modelo estadístico y de machine learning clásico utilizado para predicciones de clasificación binaria (como nuestro caso: ¿el cliente acepta o no el depósito?). 

**¿En qué casos se usa?**
- Cuando se busca un modelo **interpretable**: nos permite entender el impacto de cada variable a través de sus coeficientes.
- Como **Baseline robusto** rápido y eficiente antes de probar modelos más complejos (como Random Forest o Gradient Boosting).

**¿Por qué creemos que podría servir para este caso?**
El problema de marketing bancario requiere explicar a las áreas de negocio *por qué* se elige contactar a cierto cliente. La regresión logística ofrece una excelente explicabilidad.


In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score
from ucimlrepo import fetch_ucirepo

# Configuración Visual
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

SEED = 42
np.random.seed(SEED)


## 2. Carga de Datos y Corrección de Leakage

En el baseline (notebook 01) se identificó que la variable `duration` (duración de la llamada) es un claro ejemplo de **Target Leakage**. No podemos predecir a quién llamar usando una variable que solo conocemos *después* de que se realiza la llamada. 

Por tanto, **eliminaremos explícitamente `duration`** del dataset. Además, mantendremos la partición estratificada para evitar fugas entre Train y Test.


In [ ]:
# Cargar dataset
print("Descargando el dataset 'Bank Marketing' (ID 222)...")
bank_marketing = fetch_ucirepo(id=222)
X = bank_marketing.data.features
y = bank_marketing.data.targets

# Transformar el target a binario (0 y 1)
y = y.iloc[:, 0].replace({'no': 0, 'yes': 1})

# ELIMINAR VARIABLE DE LEAKAGE (duration)
if 'duration' in X.columns:
    X = X.drop(columns=['duration'])
    print("Variable 'duration' eliminada con éxito para evitar Target Leakage.")

# Realizamos el Train/Test Split de forma estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test: {X_test.shape}")


## 3. Preprocesado de Datos

- **Imputación/Manejo de nulos**: Trataremos 'unknown' como una categoría más.
- **Codificación de variables categóricas (One-Hot Encoding)**: Para convertir texto en variables numéricas dummy, usando `drop='first'`.
- **Escalado de variables numéricas (Standard Scaler)**: Fundamental para modelos lineales.


In [ ]:
# Identificar tipos de columnas
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Crear el pipeline de preprocesamiento
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), categorical_features)
    ])

print("Pipeline de preprocesamiento definido con éxito.")


## 4. Entrenamiento del Modelo

Entrenaremos la Regresión Logística usando un Pipeline. Utilizamos el parámetro `class_weight='balanced'` porque la variable objetivo está muy desbalanceada. Esto forzará al modelo a buscar mejor a los clientes interesados ('yes') en lugar de predecir siempre 'no'.


In [ ]:
# Definir el pipeline completo
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=SEED, class_weight='balanced', max_iter=1000))
])

# Entrenar el modelo
pipeline.fit(X_train, y_train)

print("Modelo entrenado en el escenario realista (sin duration).")


## 5. Análisis y Evaluación

Al remover `duration`, esperamos que el rendimiento caiga, pero obtendremos la métrica *real* del negocio (lo que podemos predecir a priori).


In [ ]:
# Predicciones
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

# Métricas Básicas
print("=== Reporte de Clasificación (Realista) ===")
print(classification_report(y_test, y_pred))

# AUC-ROC
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC-ROC Score: {roc_auc:.4f}")

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title("Matriz de Confusión")
plt.xlabel("Predicción")
plt.ylabel("Realidad")
plt.show()

# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc:.2f})', color='darkorange')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC')
plt.legend(loc='lower right')
plt.show()


### Comparación con el Baseline Trivíal
¿Qué pasa si predecimos siempre 'no' en este nuevo escenario?


In [ ]:
baseline_accuracy = y_test.value_counts(normalize=True).max()
model_accuracy = accuracy_score(y_test, y_pred)

print(f"Exactitud Baseline Trivíal: {baseline_accuracy:.4f}")
print(f"Exactitud Modelo Logístico Realista: {model_accuracy:.4f}")
print("\nAnálisis: La exactitud cayó fuertemente al 75.5%. Es mucho menor al 88.3% del baseline. Sin embargo, recuerda que el baseline trivial tiene un Recall de 0% en la clase positiva (nunca encuentra a un interesado). Nuestro modelo logístico encuentra al ~62% de los interesados reales, a costa de mucha precisión (llamaremos a mucha gente que dirá no).")


### Importancia de las Características
Veamos qué factores impulsan la decisión ahora que `duration` no está acaparando toda la importancia.


In [ ]:
# Extraer coeficientes y nombres de las columnas transformadas
classifier = pipeline.named_steps['classifier']
preprocessor = pipeline.named_steps['preprocessor']

cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_feature_names)

coeficientes = pd.DataFrame({
    'Feature': all_feature_names,
    'Coeficiente': classifier.coef_[0]
})

# Mostrar los 10 factores que más influyen positivamente
top_positivos = coeficientes.sort_values(by='Coeficiente', ascending=False).head(10)
# Mostrar los 10 factores que más influyen negativamente
top_negativos = coeficientes.sort_values(by='Coeficiente', ascending=True).head(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(data=top_positivos, x='Coeficiente', y='Feature', ax=axes[0], palette='Greens_r')
axes[0].set_title('Top 10 Características Positivas')

sns.barplot(data=top_negativos, x='Coeficiente', y='Feature', ax=axes[1], palette='Reds_r')
axes[1].set_title('Top 10 Características Negativas')

plt.tight_layout()
plt.show()


## 6. Conclusiones y Siguientes Pasos

**Análisis Realista:**
- **Métricas Reales**: Al quitar `duration`, el **AUC-ROC es ~0.77**, que es decente pero refleja la dificultad real de predecir comportamiento humano a priori. 
- El modelo tiene un **Recall de 62%** para la clase positiva, pero la **Precisión es de ~27%**. Esto significa que por cada 100 personas que el modelo aconseje contactar, solo 27 comprarán, pero de todos los posibles compradores en la base, lograremos capturar al 62% de ellos.
- **Factores de Decisión**: Vemos que cosas como `poutcome_success` (éxito en la campaña anterior) y el mes de contacto (`month_mar`, `month_oct`) son de los mayores indicadores positivos.

**¿Cómo usar el resto de las carpetas de este repositorio?**
Actualmente estamos metiendo todo en los `notebooks/`. Para escalar este proyecto a un nivel profesional (producción), deberíamos refactorizar usando el resto de la estructura de carpetas:
1. **`src/`**: Deberíamos extraer toda la lógica de obtención de datos, transformación y modelado en módulos de Python (`.py`). Por ejemplo, en vez de tener `train_test_split` y `ColumnTransformer` en el notebook, deberíamos tener una función `train_logistic_model(X, y)` en `src/models/train_model.py`.
2. **`data/`**: Podemos usar `data/loader.py` para abstraer `fetch_ucirepo()` en una sola función `load_bank_data()`, para que todos los notebooks compartan la misma ingesta.
3. **`figures/` y `reports/`**: En lugar de solo mostrar las gráficas ROC y de Coeficientes aquí, deberíamos guardarlas mediante script en `figures/` y generar un resumen automatizado en `reports/`. 
4. **`poster/`**: Servirá para consolidar los hallazgos principales cuando terminemos y vayamos a presentar el proyecto.
